## GPU check

In [3]:
import subprocess

subprocess.run(["nvidia-smi"], check=True)

CompletedProcess(args=['nvidia-smi'], returncode=0)

## Cloning Official Repo

In [4]:
from pathlib import Path
import subprocess

REPO = Path("/content/BoT-SORT")

if not REPO.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/NirAharon/BoT-SORT.git",
            str(REPO)
        ],
        check=True
    )

subprocess.run(
    [
        "git", "checkout",
        "251985436d6712aaf682aaaf5f71edb4987224bd"
    ],
    cwd=REPO,
    check=True
)

print("SUCCESS: BoT-SORT repository ready.")

SUCCESS: BoT-SORT repository ready.


## Create a Separate Python 3.9 Environment

In [5]:
import sys
import subprocess
from pathlib import Path

# Install the environment manager.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "uv"],
    check=True
)

# Download Python 3.9.
subprocess.run(
    [sys.executable, "-m", "uv", "python", "install", "3.9"],
    check=True
)

# Create a separate environment for BoT-SORT.
PY = "/content/botsort-env/bin/python"

if not Path(PY).exists():
    subprocess.run(
        [
            sys.executable, "-m", "uv", "venv",
            "--python", "3.9",
            "--seed",
            "/content/botsort-env"
        ],
        check=True
    )

# Display the installed Python version.
result = subprocess.run(
    [PY, "--version"],
    capture_output=True,
    text=True,
    check=True
)

print(result.stdout.strip())
print("SUCCESS: BoT-SORT Python environment is ready.")

Python 3.9.25
SUCCESS: BoT-SORT Python environment is ready.


## Install PyTorch and Torchvision

In [6]:
import subprocess

PY = "/content/botsort-env/bin/python"

# Install compatible build tools.
subprocess.run(
    [
        PY, "-m", "pip", "install",
        "pip<25",
        "setuptools<70",
        "wheel"
    ],
    check=True
)

# Install the matching PyTorch and Torchvision versions.
subprocess.run(
    [
        PY, "-m", "pip", "install",
        "torch==1.13.1+cu117",
        "torchvision==0.14.1+cu117",
        "numpy==1.23.5",
        "--extra-index-url",
        "https://download.pytorch.org/whl/cu117"
    ],
    check=True
)

# Verify that this environment can use the GPU.
verification = """
import torch
import torchvision

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

assert torch.cuda.is_available(), "GPU unavailable. Check the Colab T4 runtime."

print("GPU:", torch.cuda.get_device_name(0))
print("SUCCESS: PyTorch installation and GPU check passed.")
"""

result = subprocess.run(
    [PY, "-c", verification],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
result.check_returncode()

PyTorch: 1.13.1+cu117
Torchvision: 0.14.1+cu117
CUDA available: True
GPU: Tesla T4
SUCCESS: PyTorch installation and GPU check passed.




## Install BoT-SORT and ReID Dependencies

In [7]:
import subprocess

PY = "/content/botsort-env/bin/python"

packages = [
    "numpy==1.23.5",
    "opencv-python-headless==4.8.1.78",
    "scipy==1.10.1",
    "scikit-image==0.21.0",
    "scikit-learn==1.3.2",
    "Pillow==9.5.0",
    "Cython==0.29.36",
    "loguru",
    "tqdm",
    "thop",
    "ninja",
    "tabulate",
    "tensorboard",
    "tensorboardX",
    "lap==0.5.12",
    "motmetrics==1.4.0",
    "filterpy",
    "h5py",
    "matplotlib==3.7.5",
    "prettytable",
    "easydict",
    "PyYAML",
    "yacs",
    "termcolor",
    "gdown",
    "faiss-cpu==1.7.4",
    "pycocotools==2.0.7",
    "cloudpickle"
]

# Install tracking and appearance-model dependencies.
subprocess.run(
    [PY, "-m", "pip", "install", *packages],
    check=True
)

# Install the bounding-box overlap extension.
subprocess.run(
    [
        PY, "-m", "pip", "install",
        "--no-build-isolation",
        "cython_bbox==0.1.5"
    ],
    check=True
)

# Verify the main dependencies.
verification = """
import numpy
import cv2
import scipy
import lap
import faiss
import cython_bbox
import torch

print("NumPy:", numpy.__version__)
print("OpenCV:", cv2.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("SUCCESS: BoT-SORT dependencies installed.")
"""

result = subprocess.run(
    [PY, "-c", verification],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
result.check_returncode()

NumPy: 1.23.5
OpenCV: 4.8.1
PyTorch: 1.13.1+cu117
CUDA available: True
SUCCESS: BoT-SORT dependencies installed.




## Build the BoT-SORT Extension

In [8]:
import os
import subprocess
from pathlib import Path

PY = "/content/botsort-env/bin/python"
REPO = Path("/content/BoT-SORT")

build_env = os.environ.copy()
build_env["MAX_JOBS"] = "2"

# Compile the repository's C++ extension.
build_result = subprocess.run(
    [PY, "setup.py", "build_ext", "--inplace"],
    cwd=REPO,
    env=build_env,
    capture_output=True,
    text=True
)

print(build_result.stdout)
print(build_result.stderr)
build_result.check_returncode()

# Verify the compiled extension and tracking imports.
verification = """
from yolox import _C
from yolox.exp import get_exp
from tracker.bot_sort import BoTSORT

print("Compiled extension: OK")
print("BoT-SORT imports: OK")
print("SUCCESS: BoT-SORT build completed.")
"""

result = subprocess.run(
    [PY, "-c", verification],
    cwd=REPO,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
result.check_returncode()

running build_ext
building 'yolox._C' extension
creating build
creating build/temp.linux-x86_64-cpython-39
creating build/temp.linux-x86_64-cpython-39/content
creating build/temp.linux-x86_64-cpython-39/content/BoT-SORT
creating build/temp.linux-x86_64-cpython-39/content/BoT-SORT/yolox
creating build/temp.linux-x86_64-cpython-39/content/BoT-SORT/yolox/layers
creating build/temp.linux-x86_64-cpython-39/content/BoT-SORT/yolox/layers/csrc
creating build/temp.linux-x86_64-cpython-39/content/BoT-SORT/yolox/layers/csrc/cocoeval
cc -pthread -Wno-unused-result -Wsign-compare -Wunreachable-code -DNDEBUG -g -fwrapv -O3 -Wall -fPIC -fPIC -I/content/BoT-SORT/yolox/layers/csrc -I/content/botsort-env/lib/python3.9/site-packages/torch/include -I/content/botsort-env/lib/python3.9/site-packages/torch/include/torch/csrc/api/include -I/content/botsort-env/lib/python3.9/site-packages/torch/include/TH -I/content/botsort-env/lib/python3.9/site-packages/torch/include/THC -I/content/botsort-env/include -I/roo

## Download Detector and ReID Weights

In [9]:
import subprocess
from pathlib import Path

PY = "/content/botsort-env/bin/python"
REPO = Path("/content/BoT-SORT")
WEIGHTS = REPO / "pretrained"
WEIGHTS.mkdir(exist_ok=True)

models = [
    {
        "name": "YOLOX detector",
        "id": "1P4mY0Yyd3PPTybgZkjMYhFri88nTmJX5",
        "filename": "bytetrack_x_mot17.pth.tar"
    },
    {
        "name": "BoT-SORT ReID model",
        "id": "1QZFWpoa80rqo7O-HXmlss8J8CnS7IUsN",
        "filename": "mot17_sbs_S50.pth"
    }
]

for model in models:
    destination = WEIGHTS / model["filename"]

    if not destination.exists() or destination.stat().st_size < 1_000_000:
        print(f"Downloading: {model['name']}", flush=True)

        subprocess.run(
            [
                PY, "-m", "gdown",
                f"https://drive.google.com/uc?id={model['id']}",
                "-O", str(destination)
            ],
            check=True
        )

    assert destination.exists(), f"Missing file: {destination.name}"
    assert destination.stat().st_size > 1_000_000, (
        f"Download appears incomplete: {destination.name}"
    )

    size_mb = destination.stat().st_size / (1024 ** 2)
    print(f"Ready: {destination.name} ({size_mb:.1f} MB)")

print("\nSUCCESS: Both weight files downloaded.")

Downloading: YOLOX detector
Ready: bytetrack_x_mot17.pth.tar (756.1 MB)
Downloading: BoT-SORT ReID model
Ready: mot17_sbs_S50.pth (302.9 MB)

SUCCESS: Both weight files downloaded.


## Download and Prepare the Test Video

In [10]:
import subprocess
import urllib.request
from pathlib import Path

PY = "/content/botsort-env/bin/python"

INPUTS = Path("/content/botsort-inputs")
INPUTS.mkdir(exist_ok=True)

SOURCE_VIDEO = INPUTS / "vtest.avi"

video_url = (
    "https://raw.githubusercontent.com/"
    "opencv/opencv/master/samples/data/vtest.avi"
)

if not SOURCE_VIDEO.exists():
    print("Downloading pedestrian test video...", flush=True)
    urllib.request.urlretrieve(video_url, SOURCE_VIDEO)

prepare_video = """
import cv2

source = "/content/botsort-inputs/vtest.avi"
destination = "/content/botsort-inputs/pedestrian_test.mp4"

cap = cv2.VideoCapture(source)
assert cap.isOpened(), "Could not open the downloaded video."

fps = cap.get(cv2.CAP_PROP_FPS)
fps = fps if fps > 0 else 25.0

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

writer = cv2.VideoWriter(
    destination,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)
assert writer.isOpened(), "Could not create the test video."

count = 0

try:
    while count < 300:
        ok, frame = cap.read()
        if not ok:
            break

        writer.write(frame)
        count += 1
finally:
    cap.release()
    writer.release()

assert count > 0, "No frames were extracted."

print("Frames:", count)
print("FPS:", fps)
print("Resolution:", width, "x", height)
print("Video:", destination)
print("SUCCESS: Test video is ready.")
"""

result = subprocess.run(
    [PY, "-c", prepare_video],
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)
result.check_returncode()

Frames: 300
FPS: 10.0
Resolution: 768 x 576
Video: /content/botsort-inputs/pedestrian_test.mp4
SUCCESS: Test video is ready.




## Prepare the Demo for Colab

In [11]:
from pathlib import Path
import py_compile

REPO = Path("/content/BoT-SORT")
demo_path = REPO / "tools/demo.py"
backup_path = REPO / "tools/demo_original.py"

# Preserve the original script.
if not backup_path.exists():
    backup_path.write_text(demo_path.read_text())

# Always patch from the original so this cell can be rerun.
code = backup_path.read_text()

# Disable desktop keyboard handling for Colab.
code = code.replace("cv2.waitKey(0)", "-1")
code = code.replace("cv2.waitKey(1)", "-1")

# Update the tracker even when a frame has no detections.
old_block = """            else:
                timer.toc()
                online_im = img_info['raw_img']
"""

new_block = """            else:
                import numpy as np
                tracker.update(
                    np.empty((0, 7), dtype=np.float32),
                    img_info["raw_img"]
                )
                timer.toc()
                online_im = img_info['raw_img']
"""

assert old_block in code, "Expected video-processing block not found."
code = code.replace(old_block, new_block, 1)

# Close the video reader and writer before saving tracking records.
old_end = """        frame_id += 1

    if args.save_result:
"""

new_end = """        frame_id += 1

    cap.release()
    vid_writer.release()

    if args.save_result:
"""

assert old_end in code, "Expected video-finalization block not found."
code = code.replace(old_end, new_end, 1)

demo_path.write_text(code)

# Check Python syntax.
py_compile.compile(str(demo_path), doraise=True)

print("SUCCESS: Demo prepared for Colab.")

SUCCESS: Demo prepared for Colab.


## Run BoT-SORT + ReID on the Test Video

In [12]:
import os
import subprocess
from pathlib import Path

PY = "/content/botsort-env/bin/python"
REPO = Path("/content/BoT-SORT")

command = [
    PY,
    "-u",
    "tools/demo.py",
    "video",

    "--path",
    "/content/botsort-inputs/pedestrian_test.mp4",

    "-f",
    "yolox/exps/example/mot/yolox_x_mix_det.py",

    "-c",
    "pretrained/bytetrack_x_mot17.pth.tar",

    "--fast-reid-config",
    "fast_reid/configs/MOT17/sbs_S50.yml",

    "--fast-reid-weights",
    "pretrained/mot17_sbs_S50.pth",

    "--experiment-name",
    "colab_botsort_reid",

    "--device", "gpu",
    "--fps", "10",

    "--conf", "0.1",
    "--track_high_thresh", "0.6",
    "--track_low_thresh", "0.1",
    "--new_track_thresh", "0.7",
    "--track_buffer", "60",

    "--cmc-method", "orb",
    "--with-reid",
    "--fuse-score",
    "--fp16",
    "--fuse",
    "--save_result"
]

# Make executables from the separate environment available.
run_env = os.environ.copy()
run_env["PATH"] = (
    "/content/botsort-env/bin:"
    + run_env.get("PATH", "")
)

log_path = REPO / "tracking_run.log"

print("Starting BoT-SORT + ReID...", flush=True)

# Display progress live and save a diagnostic log.
with log_path.open("w") as log:
    process = subprocess.Popen(
        command,
        cwd=REPO,
        env=run_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)

    return_code = process.wait()

if return_code != 0:
    raise RuntimeError(
        f"Tracking failed. See the error above. "
        f"Full log: {log_path}"
    )

print("\nSUCCESS: Tracking completed.")

Starting BoT-SORT + ReID...
2026-10-28 22:15:51.707 | INFO     | __main__:main:315 - Args: Namespace(demo='video', experiment_name='colab_botsort_reid', name=None, path='/content/botsort-inputs/pedestrian_test.mp4', camid=0, save_result=True, exp_file='yolox/exps/example/mot/yolox_x_mix_det.py', ckpt='pretrained/bytetrack_x_mot17.pth.tar', device=device(type='cuda'), conf=0.1, nms=None, tsize=None, fps=10, fp16=True, fuse=True, trt=False, track_high_thresh=0.6, track_low_thresh=0.1, new_track_thresh=0.7, track_buffer=60, match_thresh=0.8, aspect_ratio_thresh=1.6, min_box_area=10, fuse_score=True, cmc_method='orb', with_reid=True, fast_reid_config='fast_reid/configs/MOT17/sbs_S50.yml', fast_reid_weights='pretrained/mot17_sbs_S50.pth', proximity_thresh=0.5, appearance_thresh=0.25, ablation=False, mot20=False)
/content/botsort-env/lib/python3.9/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Tr

## Display the Tracking Result

In [13]:
import subprocess
from pathlib import Path
from IPython.display import Video, display

OUTPUT_ROOT = Path(
    "/content/BoT-SORT/YOLOX_outputs/colab_botsort_reid"
)

# Find the most recent tracking video.
videos = sorted(
    OUTPUT_ROOT.rglob("pedestrian_test.mp4"),
    key=lambda path: path.stat().st_mtime
)

assert videos, "No tracking video found."

source = videos[-1]
destination = Path("/content/BoTSORT_ReID_Result.mp4")

# Convert to a browser-compatible video format.
result = subprocess.run(
    [
        "ffmpeg", "-y",
        "-i", str(source),
        "-c:v", "libx264",
        "-pix_fmt", "yuv420p",
        "-movflags", "+faststart",
        "-an",
        str(destination)
    ],
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print(result.stderr)

result.check_returncode()

print("SUCCESS: Result video ready.")
display(Video(str(destination), embed=True, width=900))

SUCCESS: Result video ready.


## Download the Result Video and Tracking Records

In [14]:
from pathlib import Path
import shutil
from google.colab import files

video = Path("/content/BoTSORT_ReID_Result.mp4")
output_root = Path(
    "/content/BoT-SORT/YOLOX_outputs/colab_botsort_reid"
)

assert video.exists(), "Run Cell 11 first."

# Find the tracking records belonging to the latest output video.
raw_videos = sorted(
    output_root.rglob("pedestrian_test.mp4"),
    key=lambda p: p.stat().st_mtime
)
assert raw_videos, "Tracking output not found."

latest_video = raw_videos[-1]
records = latest_video.parent.parent / (
    latest_video.parent.name + ".txt"
)
assert records.exists(), "Tracking records not found."

export_dir = Path("/content/BoTSORT_Review")
export_dir.mkdir(exist_ok=True)

shutil.copy2(video, export_dir / "tracking_video.mp4")
shutil.copy2(records, export_dir / "tracking_results.txt")

archive = shutil.make_archive(
    "/content/BoTSORT_Review",
    "zip",
    root_dir=export_dir
)

print("SUCCESS: Video and tracking records packaged.")
files.download(archive)

SUCCESS: Video and tracking records packaged.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>